
# VadCLIP — Khuếch Đại Tín Hiệu Trên Checkpoint Tự Train — Bản Kaggle

Chạy **đúng công thức đã cho 88,19**, nhưng điểm xuất phát là checkpoint
`baseline_ctrl` của hướng shift-consistency (**87,9**) — mô hình bạn tự train từ đầu chỉ
bằng trọng số CLIP — thay vì `model_ucf.pth` công bố.

## Vì sao lại là hai giai đoạn

Lần chạy trước đã thử khuếch đại **ngay từ khởi tạo ngẫu nhiên**, và nó hỏng:

| | `ctrl_scratch` (μ=1) | `class_mu12_scratch` (μ=12) |
|---|---|---|
| AUC tổng thể, đỉnh | **87,57** | **80,92** |
| AUC **lớp đích** | 60,5 | 54,2 |
| l1 cuối | 0,031 | ~0,43 |
| l2 cuối | 0,59 | ~1,78 |
| FPR@0,5 tệ nhất | 7% | **85%** |

μ = 12 làm tệ luôn cả bốn lớp mà nó được thiết kế để cứu, loss đứng yên từ epoch 3, và
tỉ lệ báo động giả trên video bình thường nổ từng cơn. Đó là chữ ký của gradient quá lớn:
μ=12 nhân với lr 2e-5 và không cắt gradient. Docstring của `_reduce` trong `losses.py`
nói thẳng rằng khuếch đại làm độ lớn gradient tăng theo μ, và đó là lý do paper ghép nó
với learning rate nhỏ hơn nhiều cùng cắt gradient chặt hơn.

Phương pháp này vốn là **giai đoạn 2**: nó nhích một nghiệm đã hội tụ, không dựng nghiệm
từ số ngẫu nhiên. Nên chuỗi đúng là:

```
CLIP  ->  shift baseline_ctrl (10 epoch, lr 2e-5, λ=0)  ->  87,9    [ĐÃ CÓ]
                     ↓  dùng làm θ'
          1 epoch, lr 2e-6, grad-clip 1.0, μ=12          ->  ?
```

Cả chuỗi chỉ xuất phát từ trọng số CLIP. `model_ucf.pth` công bố không xuất hiện ở bất kỳ
đâu trong đường huấn luyện.

## Cấu hình: chép nguyên từ lần chạy 88,19

Đây là **đúng** bộ tham số của `class_mu12_1ep`, không đổi một dòng:

```
μ = 12 · rescale-mode class · 1 epoch · lr 2e-6 · batch 64
grad-clip 1.0 · MultiStepLR([2]) · eval-steps 1280 · select-metric classifier_auc
target: Explosion, RoadAccidents, Shooting, Shoplifting
```

Ba giá trị `grad-clip`, `scheduler-milestones`, `scheduler-rate` **không truyền** — mặc
định của `ucf_option_rescale.py` đã đúng là 1.0, [2], 0.1. Đó cũng là cách notebook Colab
làm.

## Bốn lần chạy

| Tag | mode | μ | seed |
|---|---|---|---|
| `ctrl_1ep_ownsrc` | `off` | 1,0 | 234 |
| `class_mu12_1ep_ownsrc` | `class` | **12,0** | 234 |
| `ctrl_1ep_ownsrc_s1234` | `off` | 1,0 | 1234 |
| `class_mu12_1ep_ownsrc_s1234` | `class` | **12,0** | 1234 |

Mỗi lần 1 epoch: 125 bước, 13 lần chấm. Khoảng 15 phút một lần chạy, nên **cả bốn vừa
một phiên** — khác hẳn bản train-từ-đầu vốn tốn 2,3 giờ mỗi lần.

## Con số nào là kết quả

**Không phải 88,19.** Điểm xuất phát ở đây là 87,9 chứ không phải 88,02, nên đích đến
cũng khác. Hai con số cần đọc:

1. **Mức tăng so với chính θ' (87,9).** Ở lần chạy gốc, giai đoạn 2 nâng 88,02 lên 88,19,
   tức **+0,17**.
2. **Hiệu so với `ctrl_1ep_ownsrc`** — cùng θ', cùng 1 epoch, cùng luật chọn, chỉ khác μ.
   Ở lần chạy gốc hiệu này là **+0,38** ở seed 234 và **+0,14** ở seed 1234.

## ⚠️ Một điều phải ghi trong báo cáo

θ' của chuỗi này (87,9) **tự nó đã là một checkpoint chọn theo AUC tốt nhất trên tập
test**, qua khoảng 130 lần chấm. Rồi giai đoạn 2 lại chọn đỉnh lần nữa trên **cùng tập
test** đó, qua 13 lần chấm. Đây là hai tầng chọn-theo-test chồng lên nhau, và nó đẩy con
số cuối lên cao hơn thực tế.

Chuỗi 88,19 gốc chỉ có một tầng, vì `model_ucf.pth` là checkpoint của tác giả. Nêu rõ
điều này, đừng để phản biện chỉ ra.

## Chuẩn bị: đưa checkpoint 87,9 lên Kaggle

Nó nằm trong Output của phiên chạy `train_shift_consistency_kaggle.ipynb`, đường dẫn
`/kaggle/working/models/model_<tag>.pth`. Tuỳ bản notebook bạn chạy mà `<tag>` là
`baseline_ctrl` hoặc `repro_ctrl_s234` — mở tab **Output** của phiên đó mà xem tên thật.

Tải về rồi tạo một **Kaggle Dataset** chứa nó, sau đó **Add Input** vào notebook này. Mục
1 tự dò mọi file `.pth` trong `/kaggle/input`; nếu dò ra nhiều file thì điền tay
`SOURCE_OVERRIDE`.

### Code và feature

Mục 1 tự clone `vngclinh/Finetune-VadCLIP`. Feature dùng dataset công khai
`beosngu/ucf-crime-vadclip-features`. Session: GPU, Internet **On**.



## 1. Cấu Hình

Tự dò dataset trong `/kaggle/input`. Dò sai thì điền tay vào các biến `*_OVERRIDE`.

Cần **cả hai** thư mục `VadCLIP/src` và `VadCLIP/src_rescale_ewc` nằm cạnh nhau:
`_bootstrap.py` nạp `model.py`, `clip/` và `utils/` từ thư mục thứ nhất.


In [ ]:

from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

CODE_SOURCE   = 'auto'       # 'auto' | 'github' | 'dataset'
GITHUB_REPO   = 'https://github.com/vngclinh/Finetune-VadCLIP.git'
GITHUB_BRANCH = 'main'
FEATURE_DATASET_HINT = 'beosngu/ucf-crime-vadclip-features'

CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
# Checkpoint 87,9 của shift baseline_ctrl. Ví dụ:
#   SOURCE_OVERRIDE = Path('/kaggle/input/vadclip-shift-ctrl/model_baseline_ctrl.pth')
SOURCE_OVERRIDE  = None
# Tuỳ chọn, chỉ để đối chiếu ở mục 7. KHÔNG dùng để huấn luyện.
PAPER_OVERRIDE   = None


def walk_dirs(root, maxdepth=8):
    """Duyệt thư mục theo bề rộng, CÓ đi xuyên symlink.

    Không dùng rglob: `**` của pathlib gọi is_dir(follow_symlinks=False), tức nó cố ý
    bỏ qua thư mục symlink, mà Kaggle mount dataset bằng symlink.
    """
    if not root.exists():
        return
    seen, queue = set(), [(root, 0)]
    while queue:
        directory, depth = queue.pop(0)
        try:
            key = directory.resolve()
        except OSError:
            key = directory
        if key in seen:
            continue
        seen.add(key)
        yield directory
        if depth >= maxdepth:
            continue
        try:
            queue.extend((child, depth + 1)
                         for child in sorted(directory.iterdir()) if child.is_dir())
        except (PermissionError, OSError):
            pass


def find_in_input(*markers, maxdepth=8):
    for directory in walk_dirs(INPUT_ROOT, maxdepth):
        if all((directory / m).exists() for m in markers):
            return directory
    return None


def clone_repo():
    clone_dir = WORK / 'repo'
    if (clone_dir / '.git').exists():
        print('Đã có repo, cập nhật về bản mới nhất ...')
        subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1',
                        'origin', GITHUB_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard',
                        f'origin/{GITHUB_BRANCH}'], check=True)
    else:
        print('Clone', GITHUB_REPO, '...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
                        GITHUB_REPO, str(clone_dir)], check=True)
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('Commit:', sha)
    return clone_dir / 'VadCLIP'


CODE_FROM_GITHUB = False
CODE_ROOT = CODE_OVERRIDE
if CODE_ROOT is None and CODE_SOURCE != 'github':
    CODE_ROOT = find_in_input('src_rescale_ewc/ucf_train_rescale.py', 'src/model.py')
if CODE_ROOT is None and CODE_SOURCE in ('auto', 'github'):
    CODE_ROOT = clone_repo()
    CODE_FROM_GITHUB = True


def find_feature_root():
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir() and (directory / 'Vandalism').is_dir():
            return directory, 'thấy Abuse + Vandalism'
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir():
            return directory, 'chỉ thấy Abuse'
    for directory in walk_dirs(INPUT_ROOT):
        try:
            children = [d for d in directory.iterdir() if d.is_dir()]
        except (PermissionError, OSError):
            continue
        with_npy = [d for d in children if next(d.glob('*.npy'), None) is not None]
        if len(with_npy) >= 10:
            return directory, f'{len(with_npy)} thư mục con có file .npy'
    return None, None


if FEATURE_OVERRIDE is not None:
    FEATURE_ROOT, how = FEATURE_OVERRIDE, 'FEATURE_OVERRIDE đặt tay'
else:
    FEATURE_ROOT, how = find_feature_root()
print('Feature dò ra bằng:', how if FEATURE_ROOT else
      f'KHÔNG DÒ RA — Add Input dataset {FEATURE_DATASET_HINT}')

# --- Checkpoint nguồn: mọi file .pth trong /kaggle/input ---------------------------
found_pth = sorted({p for d in walk_dirs(INPUT_ROOT) for p in d.glob('*.pth')})
PAPER_MODEL = PAPER_OVERRIDE or next((p for p in found_pth if p.name == 'model_ucf.pth'), None)
candidates = [p for p in found_pth if p != PAPER_MODEL]

SOURCE_MODEL = SOURCE_OVERRIDE
if SOURCE_MODEL is None and len(candidates) == 1:
    SOURCE_MODEL = candidates[0]

print()
print('File .pth thấy trong /kaggle/input:')
for path in found_pth:
    tag = ''
    if path == PAPER_MODEL:
        tag = '   <- model_ucf.pth công bố, CHỈ để đối chiếu'
    elif path == SOURCE_MODEL:
        tag = '   <- dùng làm θ\''
    print('  ', path, tag)
if not found_pth:
    print('   (không có file .pth nào — Add Input dataset chứa checkpoint 87,9)')
elif SOURCE_MODEL is None:
    print()
    print('CÓ NHIỀU HƠN MỘT ỨNG VIÊN. Chọn tay bằng SOURCE_OVERRIDE ở đầu cell.')

# --- Code sang nơi ghi được -------------------------------------------------------
PROJECT     = WORK / 'vadclip'
SRC_DIR     = PROJECT / 'src'
RESCALE_DIR = PROJECT / 'src_rescale_ewc'
LIST_DIR    = PROJECT / 'list'
for name, destination in (('src', SRC_DIR), ('src_rescale_ewc', RESCALE_DIR),
                          ('list', LIST_DIR)):
    if not destination.exists():
        print('Copy', name, '->', destination)
        shutil.copytree(CODE_ROOT / name, destination)
for cache in PROJECT.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)

RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
MODEL_DIR  = WORK / 'models'
SCRATCH    = TEMP / 'rescale_ownsrc'
for directory in (RESULT_DIR, LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

METRICS_CSV  = str(RESULT_DIR / 'rescale_ownsrc_metrics.csv')
PERCLASS_CSV = RESULT_DIR / 'rescale_ownsrc_perclass.csv'

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ====== CẤU HÌNH — CHÉP NGUYÊN TỪ LẦN CHẠY 88,19 ======
# Nguồn: code/train_rescale_1epoch_colab.ipynb, train_1epoch().
# Khác đúng MỘT thứ: θ' là checkpoint tự train, không phải model_ucf.pth.
TARGET_CLASSES = ['Explosion', 'RoadAccidents', 'Shooting', 'Shoplifting']
MU             = 12.0
RESCALE_MODE   = 'class'
REGULARIZER    = 'none'
LAMBDA_REG     = 0.0
LAMBDA_AUTO    = 0.0
MAX_EPOCH      = 1
LR             = '2e-6'
BATCH_SIZE     = 64
NUM_WORKERS    = 4
EVAL_STEPS     = 1280            # -> chấm mỗi 10 bước -> 13 lần trong epoch
SELECT_METRIC  = 'classifier_auc'
# --grad-clip, --scheduler-milestones, --scheduler-rate KHÔNG truyền: mặc định của
# ucf_option_rescale.py đã đúng là 1.0, [2], 0.1 — cũng là cách bản Colab làm.

RUNS = [
    ('ctrl_1ep_ownsrc',             'off',   1.0, 234),
    ('class_mu12_1ep_ownsrc',       'class', 12.0, 234),
    ('ctrl_1ep_ownsrc_s1234',       'off',   1.0, 1234),
    ('class_mu12_1ep_ownsrc_s1234', 'class', 12.0, 1234),
]
ONLY_TAGS = []       # để trống = chạy tất cả
# ======================================================

sys.path.insert(0, str(RESCALE_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))     # utils/ nằm trong src/, preflight cần nó
os.chdir(RESCALE_DIR)
PY = [sys.executable, '-u']

print()
print('Nguồn code    :', 'GitHub (' + GITHUB_BRANCH + ')' if CODE_FROM_GITHUB
      else 'Dataset ' + str(CODE_ROOT))
print('Feature       :', FEATURE_ROOT)
print("θ' (nguồn)    :", SOURCE_MODEL)
print('Đối chiếu     :', PAPER_MODEL or '(không có model_ucf.pth, bỏ qua)')
print('Kết quả       :', RESULT_DIR)
print()
print('Khuếch đại    : mu =', MU, '| mode =', RESCALE_MODE,
      '| lớp đích:', ', '.join(TARGET_CLASSES))
print('Lịch          :', MAX_EPOCH, 'epoch | lr', LR,
      '| grad_clip 1.0 và milestones [2] theo mặc định script')
print('Chấm điểm     : eval_steps', EVAL_STEPS, '-> 13 lần | chọn theo', SELECT_METRIC)
print('Sẽ chạy       :', [r[0] for r in RUNS if not ONLY_TAGS or r[0] in ONLY_TAGS])


## 2. Dependencies

In [ ]:
!pip -q install ftfy regex


## 3. Nạp Sẵn Trọng Số CLIP

Không bắt buộc. Có `ViT-B-16.pt` trong Dataset thì đỡ tải 335 MB mỗi phiên.


In [ ]:

CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

source = next((d / 'ViT-B-16.pt' for d in walk_dirs(INPUT_ROOT)
               if (d / 'ViT-B-16.pt').exists()), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt. CLIP sẽ tự tải (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    print('SHA256 khớp:', hashlib.sha256(target.read_bytes()).hexdigest() == CLIP_SHA256)



## 4. Preflight

Ngoài các kiểm tra thường lệ, cell này làm một việc mà các notebook trước không cần:
**nạp thử checkpoint nguồn vào đúng kiến trúc `CLIPVAD`**.

Đó là chỗ dễ hỏng nhất của cách làm này. Checkpoint đến từ một cây code khác
(`ucf_train_augment.py`), nên nếu tên tham số hay kích thước lệch một chỗ,
`load_state_dict` sẽ báo lỗi — và tốt nhất là biết ngay bây giờ chứ không phải sau khi
đã nạp feature.

Hai điều đã kiểm sẵn dưới máy và đều khớp: bản đồ nhãn (`normal`, `abuse`, …,
`roadAccidents`) giống hệt nhau ở cả hai cây, và chín tham số kiến trúc
(`embed-dim` 512, `visual-length` 256, `visual-width` 512, `visual-head` 1,
`visual-layers` 2, `attn-window` 8, `prompt-prefix`/`postfix` 10, `classes-num` 14)
trùng nhau. Cell này xác nhận lại trên chính file thật.


In [ ]:

import csv

import numpy as np
import torch

_preflight_done = False
GT_MISSING = []


def preflight(force=False):
    global _preflight_done, GT_MISSING
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU.')
    if FEATURE_ROOT is None:
        problems.append(f'Không dò ra dataset feature ({FEATURE_DATASET_HINT}).')
    if SOURCE_MODEL is None or not Path(SOURCE_MODEL).exists():
        problems.append(
            "THIẾU checkpoint nguồn θ'. Đây là giai đoạn 2, nó tinh chỉnh TỪ một mô hình "
            'đã hội tụ. Tạo Dataset chứa model_<tag>.pth (87,9) của '
            'train_shift_consistency_kaggle.ipynb rồi Add Input, hoặc đặt SOURCE_OVERRIDE.')

    need = [RESCALE_DIR / n for n in
            ['ucf_train_rescale.py', 'ucf_option_rescale.py', 'ucf_eval_perclass.py',
             'losses.py', 'dataset_rescale.py', 'evaluation.py', '_bootstrap.py',
             'tests/test_losses.py']]
    need += [SRC_DIR / n for n in
             ['model.py', 'utils/tools.py', 'utils/layers.py',
              'utils/ucf_detectionMAP.py', 'clip/clip.py',
              'clip/bpe_simple_vocab_16e6.txt.gz']]
    need += [Path(TRAIN_LIST), Path(TEST_LIST)]
    for path in need:
        if not path.exists():
            problems.append(f'Thiếu file: {path}')

    if (RESCALE_DIR / 'ucf_option_rescale.py').exists():
        import importlib
        import ucf_option_rescale
        importlib.reload(ucf_option_rescale)
        known = {a.dest for a in ucf_option_rescale.parser._actions}
        for name in sorted({'mu', 'rescale_mode', 'target_classes', 'select_metric',
                            'grad_clip', 'use_pretrained_model'} - known):
            problems.append(f'ucf_option_rescale.py thiếu --{name.replace("_", "-")} — bản cũ.')

    try:
        from utils.layers import DistanceAdj
        probe = DistanceAdj()
        if probe(2, 32).device.type != 'cpu':
            problems.append('utils/layers.py là BẢN CŨ: DistanceAdj ghi cứng .to("cuda").')
        del probe
    except Exception as error:
        problems.append(f'Không nạp được utils/layers.py: {error}')

    GT_MISSING = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')
                  if not (LIST_DIR / n).exists()]

    if problems:
        print('PREFLIGHT KHÔNG ĐẠT:')
        for problem in problems:
            print('  -', problem)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    # --- Nạp thử checkpoint nguồn vào đúng kiến trúc --------------------------------
    from model import CLIPVAD
    import ucf_option_rescale
    args = ucf_option_rescale.parser.parse_args([])
    model = CLIPVAD(args.classes_num, args.embed_dim, args.visual_length,
                    args.visual_width, args.visual_head, args.visual_layers,
                    args.attn_window, args.prompt_prefix, args.prompt_postfix, 'cpu')
    state = torch.load(SOURCE_MODEL, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    missing, unexpected = model.load_state_dict(state, strict=False)
    real_missing = [k for k in missing if not k.startswith('clipmodel.')]
    if real_missing or unexpected:
        raise RuntimeError(
            "Checkpoint nguồn KHÔNG khớp kiến trúc.\n"
            f'  thiếu   : {real_missing[:6]}\n  thừa    : {list(unexpected)[:6]}')
    del model, state

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print("  θ' nạp thử   : khớp kiến trúc CLIPVAD, không thiếu không thừa tham số")
    print('  layers.py    : bản đã vá')
    if GT_MISSING:
        print('  Thiếu ground truth (mục 4.1 sẽ sinh lại):', [p.name for p in GT_MISSING])
    else:
        gt = np.load(LIST_DIR / 'gt_ucf.npy')
        print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')

    _preflight_done = True
    return True


preflight()



### 4.1. Sinh Lại Ground Truth (chỉ khi mục 4 báo thiếu)


In [ ]:

if GT_MISSING:
    subprocess.run([str(x) for x in PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', LIST_DIR,
    ]], check=True)
    preflight(force=True)
else:
    print('Đã có đủ ground truth, bỏ qua.')



## 5. Hàm Chạy Lệnh Và Unit Test

`build_train_cmd` chép từng cờ một từ `train_1epoch()` của notebook Colab. Khác đúng một
giá trị: `--pretrained-model-path` trỏ vào checkpoint tự train thay vì `model_ucf.pth`.

`tests/test_losses.py` kiểm chính phần khuếch đại — trọng số theo video của Eq. (1), hệ
số gradient theo lớp của Eq. (10)-(11), và điều kiện μ = 1 phải trùng khít với
`rescale_mode='off'`. Vài giây, chạy trên CPU.


In [ ]:

import time


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, rescale_mode, mu, seed):
    """Bản sao từng cờ một của train_1epoch() trong notebook Colab.

    Không truyền --grad-clip / --scheduler-milestones / --scheduler-rate: mặc định của
    ucf_option_rescale.py đã là 1.0 / [2] / 0.1, đúng thứ lần chạy 88,19 đã dùng.
    Cũng không truyền --use-pretrained-model: mặc định true, tức có nạp θ'.
    """
    return PY + [
        'ucf_train_rescale.py',
        '--pretrained-model-path', SOURCE_MODEL,
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--target-classes', *TARGET_CLASSES,
        '--seed', seed,
        '--rescale-mode', rescale_mode,
        '--mu', mu,
        '--regularizer', REGULARIZER,
        '--lambda-reg', LAMBDA_REG,
        '--lambda-auto', LAMBDA_AUTO,
        '--max-epoch', MAX_EPOCH,
        '--lr', LR,
        '--batch-size', BATCH_SIZE,
        '--num-workers', NUM_WORKERS,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', MODEL_DIR / f's2_{tag}.pth',
        '--checkpoint-path',      SCRATCH / f'checkpoint_{tag}.pth',
        '--save-cur-path',        SCRATCH / f'model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', SCRATCH / f'epoch_{tag}',
    ]


def train_1epoch(tag, rescale_mode, mu, seed):
    preflight()
    output_path = MODEL_DIR / f's2_{tag}.pth'
    if output_path.exists():
        print(f'[bỏ qua] {output_path} đã tồn tại. Xoá file nếu muốn chạy lại.')
        return None
    started = time.time()
    output = run_command(build_train_cmd(tag, rescale_mode, mu, seed),
                         log_name=f'train_{tag}.log')
    print(f'Xong sau {(time.time() - started) / 60:.1f} phút')
    return output


run_command(PY + ['tests/test_losses.py'], log_name='test_losses.log')



## 6. Bốn Lần Chạy

Mỗi lần 1 epoch: 125 bước, 13 lần chấm điểm. Khoảng 15 phút.

Dòng đầu log phải là `Loaded source model theta': ...` trỏ vào checkpoint 87,9. Nếu nó
báo `Training VadCLIP-specific layers from scratch` thì `--use-pretrained-model` đang là
false và toàn bộ mục đích của notebook này mất — dừng lại kiểm mục 1.

Cũng để ý `drift` trong các dòng log: nó đo mô hình đã đi xa θ' bao nhiêu. Ở cấu hình này
`regularizer=none` nên không có lực kéo về, `drift` chỉ là chỉ số quan sát.


In [ ]:

for tag, mode, mu, seed in RUNS:
    if ONLY_TAGS and tag not in ONLY_TAGS:
        print(f'[bỏ qua] {tag} không nằm trong ONLY_TAGS')
        continue
    print('#' * 90)
    print(f"{tag}  |  mode={mode}  mu={mu}  seed={seed}  |  θ' = {Path(SOURCE_MODEL).name}")
    print('#' * 90)
    train_1epoch(tag, mode, mu, seed)

done = [t for t, *_ in RUNS if (MODEL_DIR / f's2_{t}.pth').exists()]
print()
print('Đã có trọng số:', done)
print('Còn thiếu     :', [t for t, *_ in RUNS if t not in done])



## 7. Chấm Điểm Theo Lớp

Chấm bốn mô hình mới cộng `source` (chính θ', tức checkpoint 87,9). Nếu trong
`/kaggle/input` có `model_ucf.pth` thì chấm kèm dưới tên `paper` — **chỉ để đối chiếu**,
nó không tham gia vào đường huấn luyện nào ở đây.


In [ ]:

model_specs = [f'source={SOURCE_MODEL}']
if PAPER_MODEL and Path(PAPER_MODEL).exists():
    model_specs.append(f'paper={PAPER_MODEL}')
model_specs += [f'{tag}=' + str(MODEL_DIR / f's2_{tag}.pth') for tag, *_ in RUNS]

dropped = [s for s in model_specs if not Path(s.split('=', 1)[1]).exists()]
model_specs = [s for s in model_specs if Path(s.split('=', 1)[1]).exists()]
for spec in dropped:
    print('BỊ LOẠI (không tìm thấy file):', spec)
print('Sẽ chấm điểm', len(model_specs), 'mô hình:', [s.split('=')[0] for s in model_specs])

run_command(PY + [
    'ucf_eval_perclass.py',
    '--feature-root', FEATURE_ROOT,
    '--test-list', TEST_LIST,
    *GT_ARGS,
    '--target-classes', *TARGET_CLASSES,
    '--eval-model-paths', *model_specs,
    '--eval-output', str(PERCLASS_CSV),
], log_name='eval_perclass.log')



## 8. Bảng Kết Quả

Ba con số, theo thứ tự quan trọng:

1. **Mức tăng so với θ'** — giai đoạn 2 có nhích được mô hình lên không. Bản gốc:
   88,02 → 88,19, tức **+0,17**.
2. **Hiệu so với `ctrl_1ep_ownsrc`** — cùng θ', cùng luật chọn, chỉ khác μ. Bản gốc:
   **+0,38** ở seed 234, **+0,14** ở seed 1234.
3. **AUC lớp đích** — phương pháp nhắm vào bốn lớp này, nên nếu tổng thể tăng mà lớp đích
   không tăng thì cần giải thích.


In [ ]:

import pandas as pd

# Số của chuỗi gốc (θ' = model_ucf.pth 88,02). Để tham chiếu, KHÔNG so trực tiếp được.
ORIG = {'source': 88.02,
        'ctrl_1ep': 87.81, 'class_mu12_1ep': 88.19,
        'ctrl_1ep_s1234': 87.87, 'class_mu12_1ep_s1234': 88.01}

frame = pd.read_csv(METRICS_CSV)
frame = frame[frame.run != 'run']
for column in ('epoch', 'step', 'classifier_auc', 'target_auc_c'):
    if column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
frame = frame.drop_duplicates(subset=['run', 'epoch', 'step'], keep='last')

print('=== ĐƯỜNG CONG 13 LẦN CHẤM ===')
peaks, rows = {}, []
for tag, mode, mu, seed in RUNS:
    group = frame[frame.run == tag].sort_values('step')
    if group.empty:
        print(f'  {tag:<30} (chưa chạy)')
        continue
    best = group.loc[group.classifier_auc.idxmax()]
    peaks[tag] = float(best.classifier_auc)
    print(f'  {tag}')
    print('    ', ' '.join(f'{v:6.2f}' for v in group.classifier_auc))
    rows.append({'run': tag, 'mu': mu, 'seed': seed,
                 'dinh': round(float(best.classifier_auc), 2),
                 'dinh_o_lan_cham': int(group.classifier_auc.values.argmax()) + 1,
                 'cuoi': round(float(group.iloc[-1].classifier_auc), 2),
                 'target_auc_c_tai_dinh': round(float(best.get('target_auc_c', float('nan'))), 2)})
if rows:
    print()
    print(pd.DataFrame(rows).set_index('run').to_string())

# --- Con số chính -------------------------------------------------------------------
summary_path = Path(str(PERCLASS_CSV).replace('.csv', '_summary.csv'))
source_auc = None
if summary_path.exists():
    summary = pd.read_csv(summary_path).set_index('run')
    if 'source' in summary.index:
        source_auc = float(summary.loc['source', 'classifier_auc'])

print()
print('=' * 78)
print('KẾT QUẢ')
print('=' * 78)
if source_auc is not None:
    print(f"  θ' (checkpoint tự train)        : {source_auc:.2f}")
for run, base, seed in [('class_mu12_1ep_ownsrc', 'ctrl_1ep_ownsrc', 234),
                        ('class_mu12_1ep_ownsrc_s1234', 'ctrl_1ep_ownsrc_s1234', 1234)]:
    if run not in peaks:
        continue
    print()
    print(f'  seed {seed}')
    print(f'    μ=12               : {peaks[run]:.2f}')
    if source_auc is not None:
        print(f"    so với θ'          : {peaks[run] - source_auc:+.2f}"
              f'      (chuỗi gốc: +0,17)')
    if base in peaks:
        gap_orig = 0.38 if seed == 234 else 0.14
        print(f'    so với đối chứng   : {peaks[run] - peaks[base]:+.2f}'
              f'      (chuỗi gốc: {gap_orig:+.2f})')

print()
print('  Tham chiếu chuỗi gốc (θ\' = model_ucf.pth công bố, 88,02):')
print('    seed 234  : 88,19 so với đối chứng 87,81')
print('    seed 1234 : 88,01 so với đối chứng 87,87')
print()
print('  Nhắc: vòng trước đo được 0,58 AUC giữa hai lần chạy giống hệt nhau về mặt')
print('  toán học. Hiệu nhỏ hơn mức đó là nhiễu, không phải kết quả.')

# --- Bảng theo lớp -------------------------------------------------------------------
if not summary_path.exists():
    print()
    print('Chưa có bảng theo lớp. Chạy mục 7 trước.')
else:
    cols = [c for c in ['classifier_auc', 'target_auc_c', 'target_ap_c', 'rest_auc_c',
                        'avg_mAP', 'normal_fpr@0.5'] if c in summary.columns]
    print()
    print('=== GIÁ TRỊ TUYỆT ĐỐI THEO LỚP ===')
    print(summary[cols].round(2).to_string())

    print()
    print('=== DELTA so với đối chứng CÙNG SEED ===')
    rows = []
    for run, base in [('class_mu12_1ep_ownsrc', 'ctrl_1ep_ownsrc'),
                      ('class_mu12_1ep_ownsrc_s1234', 'ctrl_1ep_ownsrc_s1234')]:
        if run in summary.index and base in summary.index:
            rows.append({'run': run,
                         **{c: round(summary.loc[run, c] - summary.loc[base, c], 2)
                            for c in cols}})
    print(pd.DataFrame(rows).set_index('run').to_string() if rows else '(chưa đủ dữ liệu)')

    summary.to_csv(RESULT_DIR / 'rescale_ownsrc_summary.csv')
    print()
    print('Saved:', RESULT_DIR / 'rescale_ownsrc_summary.csv')



## Ghi Chú

**Chuỗi này dùng những trọng số nào.** Chỉ CLIP, từ đầu đến cuối:

```
CLIP (đóng băng)  ->  shift baseline_ctrl, 10 epoch từ đầu  ->  θ' = 87,9
                                                                  ↓
                      rescale μ=12, 1 epoch, lr 2e-6        ->  kết quả
```

`model_ucf.pth` công bố chỉ xuất hiện ở mục 7 như một dòng đối chiếu tên `paper`, không
tham gia huấn luyện.

**Vì sao không truyền `--grad-clip` và `--scheduler-milestones`.** Mặc định của
`ucf_option_rescale.py` đã là 1.0 và [2] — đúng giá trị của lần chạy 88,19. Bản
train-từ-đầu phải ghi đè chúng vì nó dùng lịch baseline; bản này thì không.

**Hai tầng chọn theo tập test.** θ' đã là checkpoint chọn theo AUC tốt nhất qua ~130 lần
chấm trên tập test, rồi giai đoạn 2 chọn đỉnh lần nữa qua 13 lần chấm trên cùng tập đó.
Chuỗi gốc chỉ có một tầng vì `model_ucf.pth` là checkpoint của tác giả. Điều này đẩy con
số cuối lên cao hơn thực tế và **phải nêu trong báo cáo**.

**Đối chứng vẫn bắt buộc.** `ctrl_1ep_ownsrc` chạy dưới đúng luật chọn đỉnh ấy. Không có
nó thì không phân biệt được phần tăng là của μ hay của việc bốc đỉnh qua 13 lần chấm.

**Kết quả của bản train-thẳng-từ-đầu vẫn nên giữ.** `class_mu12_scratch` ra 80,92 so với
87,57 của đối chứng là một phát hiện thật, không phải một lần chạy hỏng: nó cho thấy
phương pháp này không dùng được ngoài chế độ tinh chỉnh giai đoạn 2. Đó là một giới hạn
đáng viết vào báo cáo.

**CSV được nối thêm, không ghi đè.** Chạy lại một tag sẽ thêm 13 dòng mới vào
`rescale_ownsrc_metrics.csv`. Mục 8 khử trùng theo `(run, epoch, step)` và giữ lần ghi
cuối.
